# SΩI: Score-based O-INFORMATION Estimation — Quickstart

This notebook replicates the **Quickstart** from [MustaphaBounoua/soi](https://github.com/MustaphaBounoua/soi).

It clones the repository, installs dependencies, creates synthetic benchmarks (redundancy, synergy, mixed),
trains a score-based model to estimate O-information, and computes gradients of O-information.

**Reference:** *SΩI: Score-based O-INFORMATION Estimation*, Bounoua et al., ICML 2024.

## 0. Setup: Clone repo and install dependencies

In [ ]:
import os
import subprocess
import sys

# Use absolute path relative to this notebook's location
notebook_dir = os.path.dirname(os.path.abspath("__file__"))
soi_repo_dir = os.path.join("/workspaces/Cajal/soi", "soi_repo")

# Clone the SOI repository if not already present
if not os.path.isdir(soi_repo_dir):
    subprocess.run(["git", "clone", "https://github.com/MustaphaBounoua/soi.git", soi_repo_dir], check=True)
    print("Repository cloned.")
else:
    print("Repository already cloned.")

# Add the repo root to sys.path so imports work
if soi_repo_dir not in sys.path:
    sys.path.insert(0, soi_repo_dir)

print("Setup complete. soi_repo_dir =", soi_repo_dir)

## 1. Imports and default configuration

In [ ]:
from experiments.config import get_default_config
from src.benchmark.synthetic_dataset import get_task, get_dataloader
from src.libs.soi import SOI
from src.libs.soi_grad import SOI_grad
import numpy as np

## Default config
args = get_default_config()

## 2. Synthetic Benchmarks

### Redundancy benchmark
A synthetic task with 3 variables exhibiting **redundancy** (rho=0.6).

In [ ]:
## Redundancy task: 3 variables
my_settings = [{"rho": 0.6, "type": "red", "nb": 3}]
args.dim = 1
task_red = get_task(args, my_settings)
task_red.o_inf()

### Synergy benchmark
A synthetic task with 3 variables exhibiting **synergy** (rho=0.6).

In [ ]:
## Synergy task: 3 variables
my_settings = [{"rho": 0.6, "type": "syn", "nb": 3}]
args.dim = 1
task_syn = get_task(args, my_settings)
task_syn.get_summary()["o_inf"]

### Redundancy + Synergy (mixed) benchmark
A synthetic task with 6 variables: two subtasks of 3 variables each — one redundant, one synergistic.

In [ ]:
## Mixed task: 3 redundant + 3 synergistic variables
my_settings = [{"rho": 0.6, "type": "red", "nb": 3}, {"rho": 0.6, "type": "syn", "nb": 3}]
args.dim = 1
task_both = get_task(args, my_settings)
task_both.get_summary()["o_inf"]

## 3. O-information Estimation with SΩI

### General configuration
We configure the model for a quick training run with EMA, importance sampling, and weighted score functions.

In [ ]:
args.use_ema = True
args.weight_s_functions = True
args.importance_sampling = True
args.bs = 512
args.warmup_epochs = 30
args.max_epochs = 50
args.test_epoch = 5
args.debug = True
args.out_dir = "quickstart/"

# Use CPU if no GPU is available
import torch
if not torch.cuda.is_available():
    args.accelerator = "cpu"
    args.devices = 1

### Running SΩI on the redundancy benchmark

In [ ]:
gt = task_red.get_summary()
train_l, test_l = get_dataloader(task_red, args)

## Instantiate SOI
soi = SOI(args, nb_var=task_red.nb_var, gt=gt)
## Fit the model
soi.fit(train_l, test_l)
## Compute O-information using the test loader
results = soi.compute_o_inf()

print("SOI:", {"O-inf": results["o_inf"], "tc": results["tc"], "dtc": results["dtc"], "S-info": results["s_inf"]})
print("GT: ", {"O-inf": gt["o_inf"], "tc": gt["tc"], "dtc": gt["dtc"], "S-info": gt["s_inf"]})

### Running SΩI on the synergy benchmark

In [ ]:
gt = task_syn.get_summary()
train_l, test_l = get_dataloader(task_syn, args)

## Instantiate SOI
soi = SOI(args, nb_var=task_syn.nb_var, gt=gt)
## Fit the model
soi.fit(train_l, test_l)
## Compute O-information using the test loader
results = soi.compute_o_inf()

print("SOI:", {"O-inf": results["o_inf"], "tc": results["tc"], "dtc": results["dtc"], "S-info": results["s_inf"]})
print("GT: ", {"O-inf": gt["o_inf"], "tc": gt["tc"], "dtc": gt["dtc"], "S-info": gt["s_inf"]})

### Running SΩI on the mixed (redundancy + synergy) benchmark

In [ ]:
gt = task_both.get_summary()
train_l, test_l = get_dataloader(task_both, args)

## More epochs for the mixed benchmark
args.warmup_epochs = 60
args.max_epochs = 80

## Instantiate SOI
soi = SOI(args, nb_var=task_both.nb_var, gt=gt)
## Fit the model
soi.fit(train_l, test_l)
## Compute O-information using the test loader
results = soi.compute_o_inf()

print("SOI:", {"O-inf": results["o_inf"], "tc": results["tc"], "dtc": results["dtc"], "S-info": results["s_inf"]})
print("GT: ", {"O-inf": gt["o_inf"], "tc": gt["tc"], "dtc": gt["dtc"], "S-info": gt["s_inf"]})

## 4. Gradients of O-information Estimation

The gradient of O-information w.r.t. each variable reveals which variables contribute most to synergy vs. redundancy.

In [ ]:
## Ground truth gradients of O-information
gt = task_both.get_summary()
print("Ground truth g_o_inf:", " ".join([
    "x{}: {},".format(i, np.round(x, decimals=3)) for i, x in enumerate(gt["g_o_inf"])
]))

In [ ]:
## Run SOI with gradient of O-information estimation
args.warmup_epochs = 80
args.max_epochs = 100

gt = task_both.get_summary()
train_l, test_l = get_dataloader(task_both, args)

## Instantiate SOI_grad (extends SOI with gradient estimation)
soi = SOI_grad(args, nb_var=task_both.nb_var, gt=gt)
## Fit the model
soi.fit(train_l, test_l)
## Compute O-information with gradients
print("Final estimation:", soi.compute_o_inf_with_grad())